# Consulta masiva de NIT en RUES (rues.org.co)

Lee una lista de NIT desde Excel y, para cada uno, consulta **https://www.rues.org.co/** con Chrome (Selenium) y extrae:

1. **Información general**: identificación, categoría, tipo de sociedad, organización, cámara, matrícula, fechas, estado...
2. **Actividad económica**: códigos CIIU y descripción.
3. **Representante legal**: principal(es) y suplente(s) con cargo, nombre e identificación, más el texto completo.

**Entrada:** `entrada/nits.xlsx` (columna `NIT`).  
**Salida:** `salida/resultado_rues.xlsx` con las hojas `InfoGeneral`, `Actividades`, `Representantes` y `Errores`.

**Notas**
- Acepta NIT **con o sin dígito de verificación** (`901235729`, `9012357298`, `901235729-8`, `901.235.729-8`).
- El avance se guarda en `salida/progreso/resultados.jsonl`; si se interrumpe, vuelva a ejecutar y continúa donde iba.
- RUES no carga en modo *headless* (página en blanco): el navegador debe quedar **visible**. No lo cierre mientras corre.
- La pestaña *Representante legal* es texto libre copiado del certificado; el script reconoce los formatos de tabla
  (`CARGO / NOMBRE / IDENTIFICACIÓN`) y de lista (`REPRESENTACION LEGAL (PRINCIPALES)`). Si una cámara usa otro formato,
  el texto completo queda en `texto_representantes` para revisarlo.

## 1. Instalación (solo la primera vez)

In [ ]:
%pip install -q selenium pandas openpyxl

## 2. Configuración

In [ ]:
ARCHIVO_ENTRADA   = "entrada/nits.xlsx"  # Excel con la lista de NIT
HOJA              = 0                    # nombre o índice de la hoja
COLUMNA_NIT       = "NIT"                # nombre de la columna con los NIT
ARCHIVO_SALIDA    = "salida/resultado_rues.xlsx"
CARPETA_PROGRESO  = "salida/progreso"    # avance + capturas de error

PAUSA_MIN, PAUSA_MAX = 2.0, 4.0          # espera aleatoria entre NIT (segundos)
TIMEOUT           = 45                   # espera máxima por página (segundos)
REINTENTOS        = 2                    # reintentos por NIT si algo falla
TODAS_LAS_MATRICULAS = False             # False = solo la matrícula principal activa
                                         # True  = todas las matrículas distintas del NIT
MAX_REP_EN_COLUMNAS = 4                  # representantes que se copian como columnas en InfoGeneral
GUARDAR_EXCEL_CADA = 20                  # reescribe el Excel cada N NIT
LIMITE            = None                 # p.ej. 5 para probar con pocos NIT

## 3. Librerías

In [ ]:
import json, random, re, time
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

Path(CARPETA_PROGRESO).mkdir(parents=True, exist_ok=True)
Path(ARCHIVO_SALIDA).parent.mkdir(parents=True, exist_ok=True)
ARCHIVO_PROGRESO = Path(CARPETA_PROGRESO) / "resultados.jsonl"

## 4. Limpieza del NIT y dígito de verificación (DV)

- Con guion (`901235729-8`): NIT = antes del guion, DV = después.
- 10 dígitos que empiezan por 8 o 9 y cuyo último dígito **coincide** con el DV calculado de los primeros 9: se separa el DV.
- En otro caso se toma completo como NIT (p.ej. cédulas de personas naturales) y se calcula el DV.

In [ ]:
PESOS_DIAN = [71, 67, 59, 53, 47, 43, 41, 37, 29, 23, 19, 17, 13, 7, 3]

def calcular_dv(nit: str) -> int:
    nit = str(nit).zfill(15)
    r = sum(int(d) * p for d, p in zip(nit, PESOS_DIAN)) % 11
    return r if r in (0, 1) else 11 - r

def limpiar_nit(valor):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None, None
    s = re.sub(r"\.0$", "", str(valor).strip())
    s = s.replace(".", "").replace(",", "").replace(" ", "")
    m = re.match(r"^(\d+)[-‐–—](\d)$", s)
    if m:
        return m.group(1), m.group(2)
    dig = re.sub(r"\D", "", s)
    if not dig:
        return None, None
    if len(dig) == 10 and dig[0] in "89" and calcular_dv(dig[:9]) == int(dig[-1]):
        return dig[:9], dig[-1]
    return dig, str(calcular_dv(dig))

for x in ["901235729", "9012357298", "901235729-8", "901.235.729-8", "1020304050"]:
    print(f"{x:>15} -> {limpiar_nit(x)}")

## 5. Leer la lista de NIT

In [ ]:
df_in = pd.read_excel(ARCHIVO_ENTRADA, sheet_name=HOJA, dtype=str)
if COLUMNA_NIT not in df_in.columns:
    raise ValueError(f"No encuentro la columna '{COLUMNA_NIT}'. Columnas: {list(df_in.columns)}")

df_in[["nit", "dv"]] = df_in[COLUMNA_NIT].apply(lambda v: pd.Series(limpiar_nit(v)))
invalidos = df_in[df_in["nit"].isna()]
nits = df_in.dropna(subset=["nit"]).drop_duplicates("nit")[[COLUMNA_NIT, "nit", "dv"]]
if LIMITE:
    nits = nits.head(LIMITE)
print(f"NIT válidos (sin duplicados): {len(nits)}  |  inválidos/vacíos: {len(invalidos)}")
nits.head()

## 6. Lectura del texto de representantes legales

Convierte el texto del certificado en filas `tipo (PRINCIPAL/SUPLENTE) | cargo | nombre | tipo_id | identificacion | dv`.

In [ ]:
import re

ID_RE = re.compile(
    r"(?<![A-Za-zÁ-ú])(?P<tipo>C\.?\s?C\.?|C\.?\s?E\.?|N\.?\s?I\.?(?:\s?T\.?)?|P\.?\s?P\.?|T\.?\s?I\.?|PAS(?:APORTE)?\.?)"
    r"\s*(?:No\.?|N[°º]\.?|#)?\s*(?P<num>\d[\d\.]*\d|\d)(?:\s*-?\s*(?P<dv>\d))?\s*$",
    re.I)
LISTA_RE = re.compile(r"^\s*(?P<num>\d[\d\.]{4,})\s*-\s*(?P<nombre>[^\d].+?)\s*$")
CORTE_RE = re.compile(r"^\s*(\.|por\s|cargo\b|certifica|facultades|mediante|que\s|el\s|la\s|los\s|las\s)", re.I)
VOCAB_CARGO = {"REPRESENTANTE", "LEGAL", "PRINCIPAL", "SUPLENTE", "SUPLENTES", "PRESIDENTE", "VICEPRESIDENTE",
               "GERENTE", "SUBGERENTE", "PRIMER", "SEGUNDO", "TERCER", "CUARTO", "QUINTO", "DEL", "DE", "Y",
               "GENERAL", "PARA", "ASUNTOS", "JUDICIALES", "DIRECTOR", "DIRECTORA", "EJECUTIVO", "ADMINISTRADOR",
               "LIQUIDADOR", "APODERADO", "SECRETARIO", "TESORERO", "MIEMBRO", "JUNTA", "DIRECTIVA"}

def _tipo_id(t):
    t = re.sub(r"[\.\s]", "", t.upper())
    return {"CC": "CC", "CE": "CE", "NIT": "NIT", "NI": "NIT", "PP": "PP", "TI": "TI"}.get(t, "PASAPORTE" if t.startswith("PAS") else t)

def _cortar(seg, col):
    # Divide una línea en (izquierda, derecha) en la columna col si cae en un espacio
    if col and len(seg) > col and seg[col - 1] == " ":
        return seg[:col], seg[col:]
    if col and len(seg) <= col:
        return seg, ""
    # línea desalineada: tomar palabras iniciales que sean de cargo
    pal = seg.split()
    k = 0
    while k < len(pal) and pal[k].upper().strip(",.") in VOCAB_CARGO:
        k += 1
    return " ".join(pal[:k]), " ".join(pal[k:])

def _limpia(s):
    return re.sub(r"\s+", " ", s or "").strip(" .,-")

def parsear_representantes(texto):
    # Extrae [{tipo, cargo, nombre, tipo_id, identificacion, dv}] del texto de la pestaña Representante legal
    if not texto:
        return []
    lineas = [l.replace("\xa0", " ").rstrip() for l in texto.replace("\r", "").split("\n")]
    personas = []
    col_nom = col_id = None
    seccion = None          # para el formato lista: PRINCIPAL / SUPLENTE
    actual = None
    alterna = False         # formato "Cargo/Nombre": línea de cargo y debajo nombre + identificación
    cargo_pend = ""

    def cerrar():
        nonlocal actual
        if actual:
            actual["cargo"] = _limpia(actual["cargo"])
            actual["nombre"] = _limpia(actual["nombre"])
            if actual["nombre"]:
                c = (actual["cargo"] + " " + (actual.get("seccion") or "")).lower()
                actual["tipo"] = "SUPLENTE" if "suplente" in c else "PRINCIPAL"
                actual.pop("seccion", None)
                personas.append(actual)
        actual = None

    for l in lineas:
        s = l.strip()
        up = s.upper()
        # formato alterno (p.ej. Barranquilla): "Cargo/Nombre      Identificación"
        if re.match(r"^CARGO\s*/\s*NOMBRE", up):
            cerrar()
            alterna, cargo_pend = True, ""
            continue
        if alterna:
            m = ID_RE.search(l)
            if m:
                personas.append({"cargo": _limpia(cargo_pend), "nombre": _limpia(l[:m.start()]),
                                 "tipo_id": _tipo_id(m.group("tipo")),
                                 "identificacion": m.group("num").replace(".", ""), "dv": m.group("dv") or "",
                                 "tipo": "SUPLENTE" if "suplente" in cargo_pend.lower() else "PRINCIPAL"})
                cargo_pend = ""
            elif s and len(s) >= 50 and not re.search(r"\S\s{3,}\S", s):
                alterna = False      # empieza un párrafo: fin de la tabla
            elif s:
                cargo_pend = s
            continue
        # encabezado de tabla
        if re.match(r"^CARGO\s+NOMBRE", up):
            cerrar()
            col_nom = l.upper().find("NOMBRE")
            m = re.search(r"IDENTIFICACI", l.upper())
            col_id = m.start() if m else None
            continue
        # encabezados del formato lista
        if len(s) < 70 and (re.search(r"REPRESENTACI[OÓ]N\s+LEGAL\s*\(\s*(PRINCIPAL|SUPLENTE)", up)
                            or re.match(r"^\*{0,4}\s*(PRINCIPALES|SUPLENTES)\b", up)):
            cerrar()
            seccion = "SUPLENTE" if "SUPLENTE" in up else "PRINCIPAL"
            continue
        if re.search(r"FACULTADES|^CERTIFICA", up):
            cerrar()
            seccion = None if "FACULTADES" in up else seccion
            continue
        # formato lista: 1083881604 - SILVA VERA YAMID
        m = LISTA_RE.match(s)
        if m and seccion:
            cerrar()
            num = m.group("num").replace(".", "")
            personas.append({"tipo": seccion, "cargo": f"REPRESENTANTE LEGAL {seccion}", "nombre": _limpia(m.group("nombre")),
                             "tipo_id": "", "identificacion": num, "dv": ""})
            continue
        # formato tabla: línea con identificación al final
        m = ID_RE.search(l)
        if m and col_nom is not None:
            cerrar()
            izq = l[:m.start()].rstrip()
            cargo, nombre = _cortar(izq, col_nom)
            actual = {"cargo": cargo, "nombre": nombre, "tipo_id": _tipo_id(m.group("tipo")),
                      "identificacion": m.group("num").replace(".", ""), "dv": m.group("dv") or "",
                      "seccion": seccion}
            continue
        # continuación de la persona actual
        if actual is not None:
            if not s or CORTE_RE.match(s) or len(s) > (col_id or 50) + 10:
                cerrar()
                continue
            seg = l[:col_id] if col_id else l
            cargo, nombre = _cortar(seg, col_nom)
            actual["cargo"] += " " + cargo
            actual["nombre"] += " " + nombre
    cerrar()
    return [{k: p.get(k, "") for k in ["tipo", "cargo", "nombre", "tipo_id", "identificacion", "dv"]} for p in personas]

## 7. Navegador y extracción en RUES

In [ ]:
URL_BUSQUEDA = "https://www.rues.org.co/buscar/RM/{nit}"
SEL_PANEL = "#detail-tabs-tabpane-pestana_{}"
SEL_TAB   = "#detail-tabs-tab-pestana_{}"

def crear_driver():
    opts = webdriver.ChromeOptions()
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--lang=es-CO")
    opts.add_argument("--disable-notifications")
    d = webdriver.Chrome(options=opts)        # Selenium Manager descarga el driver solo
    d.set_page_load_timeout(60)
    return d

def pausa(a=PAUSA_MIN, b=PAUSA_MAX):
    time.sleep(random.uniform(a, b))

def cerrar_popups(driver):
    # Cierra el aviso emergente (SweetAlert) que RUES muestra al cargar
    try:
        driver.execute_script("document.querySelectorAll('.swal2-close').forEach(b => b.click());")
    except WebDriverException:
        pass

def texto_body(driver):
    try:
        return driver.find_element(By.TAG_NAME, "body").text
    except WebDriverException:
        return ""

JS_TARJETAS = r'''
return [...document.querySelectorAll('.card-result')].map(c => {
  const ls = [...c.querySelectorAll('a')].filter(a => /ver informaci/i.test(a.innerText));
  const a = ls.find(x => x.offsetParent !== null) || ls[0];
  return a ? [a, c.innerText] : null;
}).filter(Boolean);
'''

JS_PARES = r'''
const p = document.querySelector(arguments[0]);
if (!p) return [];
return [...p.querySelectorAll('.registroapi')].map(r => [
  ((r.querySelector('.registroapi__etiqueta') || {}).innerText || '').trim(),
  ((r.querySelector('.registroapi__valor') || {}).innerText || '').trim()
]).filter(x => x[0] || x[1]);
'''

JS_TITULO = r'''
const h = [...document.querySelectorAll('h1, h2, h3')].filter(e => e.offsetParent !== null)
  .map(e => e.innerText.trim()).filter(t => t && !/registro mercantil|^rues$/i.test(t));
return h[0] || '';
'''

def buscar(driver, nit):
    # Devuelve [(link, texto_tarjeta)] cuyas tarjetas tienen exactamente este NIT; [] si no hay.
    # RUES también busca por nombre, por eso se filtra por la identificación de la tarjeta.
    patron = re.compile(rf"Identificaci[oó]n\s*{nit}(?!\d)", re.I)
    for carga in range(2):                      # si la página queda en blanco, recarga una vez
        driver.get(URL_BUSQUEDA.format(nit=nit))
        t0, n_prev, sin_tarjetas = time.time(), -1, None
        while time.time() - t0 < TIMEOUT:
            cerrar_popups(driver)
            tarjetas = driver.execute_script(JS_TARJETAS)
            if tarjetas and len(tarjetas) == n_prev:          # cantidad estable entre dos lecturas
                return [t for t in tarjetas if patron.search(t[1])]
            n_prev = len(tarjetas)
            if not tarjetas and "Filtrar por" in texto_body(driver):
                sin_tarjetas = sin_tarjetas or time.time()
                if time.time() - sin_tarjetas > 6:            # página cargada y sin tarjetas
                    return []
            time.sleep(1)
    raise TimeoutException("La búsqueda no cargó (página en blanco)")

def puntuar(texto):
    t = texto.lower()
    p = 0
    if re.search(r"estado\s*\n?\s*activ", t): p += 3
    if re.search(r"cancelad|inactiv|liquidad", t): p -= 2
    if re.search(r"establecimiento|sucursal|agencia", t): p -= 5
    if re.search(r"persona juridica principal|persona natural|sociedad", t): p += 1
    return p

JS_PANEL_LISTO = r'''
const p = document.querySelector(arguments[0]);
if (!p || p.querySelector('.spinner-border, .spinner-grow')) return false;
return p.innerText.trim().length > 0 && !/^cargando/i.test(p.innerText.trim());
'''

def esperar_panel(driver, clave, timeout=TIMEOUT):
    # Clic en la pestaña y espera a que su panel termine de cargar (sin spinner)
    try:
        driver.execute_script(f"document.querySelector('{SEL_TAB.format(clave)}')?.click();")
    except WebDriverException:
        pass
    t0 = time.time()
    while time.time() - t0 < timeout:
        if driver.execute_script(JS_PANEL_LISTO, SEL_PANEL.format(clave)):
            time.sleep(0.5)
            return True
        time.sleep(0.5)
    return False

def leer_detalle(driver, link, tarjeta):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'}); arguments[0].click();", link)
    WebDriverWait(driver, TIMEOUT, 0.5).until(lambda d: d.execute_script(
        f"return document.querySelectorAll('{SEL_PANEL.format('general')} .registroapi__valor').length"))
    time.sleep(1)
    cerrar_popups(driver)

    general = driver.execute_script(JS_PARES, SEL_PANEL.format("general"))
    razon = driver.execute_script(JS_TITULO)

    esperar_panel(driver, "economica")
    actividades = [{"ciiu": e, "descripcion": v}
                   for e, v in driver.execute_script(JS_PARES, SEL_PANEL.format("economica")) if e or v]

    if not esperar_panel(driver, "representante"):
        raise TimeoutException("La pestaña Representante legal no terminó de cargar")
    texto_rep = driver.execute_script(
        f"const p=document.querySelector('{SEL_PANEL.format('representante')}');"
        "if(!p) return ''; const l=p.querySelector('.legal'); return (l||p).innerText;") or ""
    reps = parsear_representantes(texto_rep)

    return {"razon_social": razon, "tarjeta": tarjeta, "general": general,
            "actividades": actividades, "representantes": reps, "texto_representantes": texto_rep}

def consultar_nit(driver, nit):
    tarjetas = buscar(driver, nit)
    if not tarjetas:
        return {"estado": "SIN_RESULTADOS", "n_resultados": 0, "matriculas": []}
    # quitar tarjetas repetidas (RUES a veces muestra la misma matrícula varias veces)
    vistos, unicas = set(), []
    for i, (_, txt) in enumerate(tarjetas):
        if txt not in vistos:
            vistos.add(txt); unicas.append(i)
    orden = sorted(unicas, key=lambda i: -puntuar(tarjetas[i][1]))
    if not TODAS_LAS_MATRICULAS:
        orden = orden[:1]
    matriculas = []
    for k, idx in enumerate(orden):
        if k > 0:
            pausa(1, 2)
            tarjetas = buscar(driver, nit)
        link, txt = tarjetas[idx]
        matriculas.append(leer_detalle(driver, link, txt))
    return {"estado": "OK", "n_resultados": len(unicas), "matriculas": matriculas}

## 8. Prueba con un NIT (recomendado antes de la corrida masiva)

In [ ]:
NIT_PRUEBA = "901235729"

d = crear_driver()
try:
    res = consultar_nit(d, limpiar_nit(NIT_PRUEBA)[0])
finally:
    d.quit()

print("Estado:", res["estado"], "| matrículas distintas:", res["n_resultados"])
for m in res["matriculas"]:
    print("\nRazón social:", m["razon_social"])
    for k, v in m["general"]:
        print(f"  {k}: {v}")
    print("Actividades:", m["actividades"])
    print("Representantes:")
    for r in m["representantes"]:
        print("  ", r)

## 9. Funciones para armar el Excel

In [ ]:
def construir_tablas(hechos):
    info, acts, reps, errs = [], [], [], []
    for nit, r in hechos.items():
        base = {"nit_original": r.get("nit_original"), "nit": nit, "dv": r.get("dv"),
                "estado_consulta": r["estado"], "matriculas_encontradas": r.get("n_resultados")}
        if r["estado"] != "OK":
            info.append(base)
            errs.append({**base, "detalle": r.get("error", "Sin resultados en RUES")})
            continue
        for m_i, m in enumerate(r["matriculas"], 1):
            fila = {**base, "matricula_n": m_i, "razon_social": m["razon_social"]}
            for k, v in m["general"]:
                col = k
                while col in fila:
                    col += "_"
                fila[col] = v
            for j, a in enumerate(m["actividades"], 1):
                fila[f"ciiu_{j}"] = a["ciiu"]
                fila[f"ciiu_{j}_descripcion"] = a["descripcion"]
                acts.append({"nit": nit, "dv": r.get("dv"), "razon_social": m["razon_social"],
                             "matricula_n": m_i, "orden": j, **a})
            fila["n_representantes"] = len(m["representantes"])
            for j, p in enumerate(m["representantes"], 1):
                reps.append({"nit": nit, "dv": r.get("dv"), "razon_social": m["razon_social"],
                             "matricula_n": m_i, "orden": j, **p})
                if j <= MAX_REP_EN_COLUMNAS:
                    for k in ["tipo", "cargo", "nombre", "tipo_id", "identificacion"]:
                        fila[f"rep{j}_{k}"] = p[k]
            fila["texto_representantes"] = (m["texto_representantes"] or "")[:32000]   # límite de celda Excel
            info.append(fila)
    return pd.DataFrame(info), pd.DataFrame(acts), pd.DataFrame(reps), pd.DataFrame(errs)

def exportar_excel(hechos, ruta=ARCHIVO_SALIDA):
    tablas = construir_tablas(hechos)
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        for df, hoja in zip(tablas, ["InfoGeneral", "Actividades", "Representantes", "Errores"]):
            df.to_excel(xw, sheet_name=hoja, index=False)
    return tablas

def cargar_progreso():
    hechos = {}
    if ARCHIVO_PROGRESO.exists():
        for linea in ARCHIVO_PROGRESO.read_text(encoding="utf-8").splitlines():
            if linea.strip():
                r = json.loads(linea)
                hechos[r["nit"]] = r          # la última línea de cada NIT es la vigente
    return hechos

def guardar_progreso(reg):
    with open(ARCHIVO_PROGRESO, "a", encoding="utf-8") as f:
        f.write(json.dumps(reg, ensure_ascii=False) + "\n")

## 10. Corrida masiva (con reanudación)

Salta los NIT ya consultados y reintenta los que quedaron en `ERROR`. Puede detenerla (botón ■) y volver a ejecutarla.

In [ ]:
hechos = cargar_progreso()
pendientes = [r for r in nits.itertuples(index=False)
              if r.nit not in hechos or hechos[r.nit]["estado"] == "ERROR"]
print(f"Ya procesados: {len(hechos)}  |  Pendientes: {len(pendientes)}")

driver = crear_driver()
try:
    for i, fila in enumerate(pendientes, 1):
        nit, dv, original = fila[1], fila[2], fila[0]
        reg = None
        for intento in range(1, REINTENTOS + 2):
            try:
                reg = {"nit": nit, "dv": dv, "nit_original": original, **consultar_nit(driver, nit)}
                break
            except Exception as e:
                reg = {"nit": nit, "dv": dv, "nit_original": original, "estado": "ERROR",
                       "error": f"{type(e).__name__}: {str(e)[:300]}", "n_resultados": None, "matriculas": []}
                try:
                    driver.save_screenshot(str(Path(CARPETA_PROGRESO) / f"error_{nit}_{intento}.png"))
                except Exception:
                    pass
                if isinstance(e, WebDriverException) and re.search(r"invalid session|no such window|disconnected", str(e), re.I):
                    try: driver.quit()
                    except Exception: pass
                    driver = crear_driver()
                pausa(3, 6)
        hechos[nit] = reg
        guardar_progreso(reg)
        extra = ""
        if reg["estado"] == "OK":
            m = reg["matriculas"][0]
            extra = f"{m['razon_social'][:40]} | {len(m['actividades'])} CIIU | {len(m['representantes'])} rep."
        elif reg["estado"] == "ERROR":
            extra = reg["error"][:80]
        print(f"[{i}/{len(pendientes)}] {nit}-{dv}: {reg['estado']}  {extra}")
        if i % GUARDAR_EXCEL_CADA == 0:
            exportar_excel(hechos)
        pausa()
finally:
    driver.quit()
    exportar_excel(hechos)
    print("Excel actualizado:", ARCHIVO_SALIDA)

## 11. Revisar resultados

In [ ]:
df_info, df_act, df_rep, df_err = exportar_excel(cargar_progreso())
print(f"{len(df_info)} filas InfoGeneral | {len(df_act)} actividades | {len(df_rep)} representantes | {len(df_err)} errores/sin resultado")
df_info.head()

In [ ]:
df_rep.head(20)

In [ ]:
df_act.head(20)

In [ ]:
# NIT consultados OK pero sin representantes reconocidos: revisar la columna texto_representantes
if len(df_info) and "n_representantes" in df_info:
    display(df_info[(df_info["estado_consulta"] == "OK") & (df_info["n_representantes"] == 0)]
            [["nit", "razon_social", "texto_representantes"]])